# <font color ='green'> Gemma cross-validation fine tuning with classification head

### import libraries

Use transformer_venv because it has transformer v4.57 which works, script breaks with transformer v5 as they have changed how Trainer works

In [ ]:
import json
import torch
import pandas as pd
from datasets import Dataset, load_dataset
from huggingface_hub import login
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from transformers import (
    LongformerForSequenceClassification,
    LongformerTokenizerFast,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

import os
import gc
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold
import numpy as np
import matplotlib.pyplot as plt
import bitsandbytes as bnb
import re
import evaluate
from datetime import datetime
import time
from typing import Optional, Dict, Any, List
from sklearn.utils.class_weight import compute_class_weight
from torch.nn import CrossEntropyLoss

from dotenv import dotenv_values
config = dotenv_values(".env")  
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# Data cleaning function - combined
def standardization(sent: str) -> str:
    '''
    Input: raw reviews (string)
    Output: cleaned & standardized reviews (string)
    '''
    # Convert to lowercase, remove unwanted patterns, and remove non-alphanumeric characters
    sent = re.sub(r'[^0-9a-zA-Z-ZäöüÄÖÜßéóƒÚâèåèñéçýáúåí\s]', '', sent.lower())  
    # Remove specific MAUDE patterns
    sent = re.sub(r'\(b\)\(6\)|\(b\) \(6\)|\(b\)\(4\)|\(b\) \(4\)|\[rs\]\.\\n', '', sent)
    # Replace multiple spaces and strip leading/trailing whitespaces
    sent = re.sub(r'\s+', ' ', sent).strip()  
    
    return sent

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Input: DataFrame with columns 'text' and 'label'
    Output: Cleaned DataFrame with standardized text
    '''
    # Standardize text by applying the standardization function to each row
    df["text"] = df["text"].apply(standardization) 

    return df

In [ ]:
metric = evaluate.combine(["accuracy", "f1", "precision", "recall"])

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probabilities = torch.nn.functional.softmax(torch.tensor(predictions), dim=1).numpy()
    
    predictions = np.argmax(predictions, axis=1)  # Convert probabilities to predicted labels [0 or 1]
    roc_auc = roc_auc_score(labels, probabilities[:, 1])  # Use probabilities of the positive clas
    print(roc_auc)

    dict_metric = metric.compute(predictions=predictions, references=labels)
    dict_metric.update({'roc_auc': roc_auc})
    return dict_metric

In [ ]:
########################################################################

## Model Parameters
learning_rate = 5e-5
num_train_epochs = 5
weight_decay = 0.01
early_stop_patience = 2 # increase a bit
early_stop_threshold = 0.01 # increase a bit
max_words            = 2000
#early_stopping = EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.0)

tokenizer_maxlength = 2048 
attn_win = 512
# Model name
model_name_str = "allenai/longformer-base-4096"


## Output folder
# Define model name and timestamp
unique_model_str = 'Bert-longformer-weighted' # CHANGE PER SCRIPT
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M')  # Ensures uniqueness

# Define output folders
output_base_path = 'results/'
unique_name_date = f"{unique_model_str}_early_stop_{early_stop_patience}_{early_stop_threshold}_max_n_{max_words}_{timestamp}"
output_folder = os.path.join(output_base_path, unique_name_date)

os.makedirs(output_folder, exist_ok=True) # Create directories
print(f"Model results will be saved in: {output_folder}") # Print directories for verification


# TO DO LATER: delete checkpoints

## Load and preprocess dataset
data_folder = 'data/'
data_file = 'cybersecurity_annotated_data.pq'
df = pd.read_parquet(os.path.join(data_folder, data_file))
df = clean_dataframe(df[['ID', 'text', 'label']].dropna())
df['text'] = df['text'].apply(lambda x : ' '.join(x.split(' ')[:max_words]))
df.head()

In [ ]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenizer = LongformerTokenizerFast.from_pretrained(model_name_str, max_length=tokenizer_maxlength) 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
class ModelHandler:
    """
    Eine Klasse zum Verwalten von Transformers-Modellen mit LoRA-Adaptern.
    
    Diese Klasse ermöglicht:
    - Einmaliges Laden eines Basis-Modells
    - Hinzufügen von LoRA-Adaptern
    - Zurücksetzen der Adapter-Gewichte
    - Erneutes Training des Modells
    """
    
    def __init__(self, 
                 model_name: str,
                 num_labels: int = 2,
                 torch_dtype: torch.dtype = torch.bfloat16,
                 device_map: str = "auto",
                 id2label: Optional[Dict[int, str]] = None,
                 label2id: Optional[Dict[str, int]] = None,
                 trust_remote_code: bool = True,
                 attn_implementation: str = "eager"):
        """
        Initialisiert den ModelHandler.
        
        Args:
            model_name: Name oder Pfad des zu ladenden Modells
            num_labels: Anzahl der Labels für Klassifikation
            torch_dtype: Datentyp für das Modell
            device_map: GPU-Device-Mapping
            id2label: Mapping von IDs zu Labels
            label2id: Mapping von Labels zu IDs
            trust_remote_code: Vertrauen in Remote-Code
            attn_implementation: Attention-Implementierung
        """
        self.model_name = model_name
        self.num_labels = num_labels
        self.torch_dtype = torch_dtype
        self.device_map = device_map
        self.trust_remote_code = trust_remote_code
        self.attn_implementation = attn_implementation
        
        # Standard-Labels falls nicht angegeben
        self.id2label = id2label or {0: 'noCyber', 1: 'Cyber'}
        self.label2id = label2id or {'noCyber': 0, 'Cyber': 1}
        
        # Model-Zustand verfolgen
        self.base_model = None
        self.model = None
        self.lora_config = None
        self.has_adapters = False
        self.original_state_dict = None
        
        # Model laden
        self._load_base_model()
        
    def _load_base_model(self) -> None:
        """Lädt das Basis-Modell."""
        try:
            self.base_model = LongformerForSequenceClassification.from_pretrained(
                self.model_name,
                num_labels=self.num_labels,
                torch_dtype=self.torch_dtype,
                # use_cache=False,               # not valid for longformer
                id2label=self.id2label,
                label2id=self.label2id,
                trust_remote_code=self.trust_remote_code,
                attn_implementation=self.attn_implementation,
                device_map=self.device_map,
                attention_window=attn_win,
            )
            
            # Gradient Checkpointing aktivieren
            self.base_model.gradient_checkpointing_enable()
            
            # Für k-bit Training vorbereiten
            # Not using a quantized model
            # self.base_model = prepare_model_for_kbit_training(self.base_model)
            
            # Aktuelle Model-Referenz setzen
            self.model = self.base_model
            
            # Original-Zustand speichern für Reset
            #self.original_state_dict = copy.deepcopy(self.base_model.state_dict())
            
        except Exception as e:
            raise
    
    def add_lora_adapters(self,
                         r: int = 16,
                         lora_alpha: int = 32,
                         lora_dropout: float = 0.1,
                         target_modules: Optional[List[str]] = None,
                         bias: str = "none") -> None:
        """
        Fügt LoRA-Adapter zum Modell hinzu.
        
        Args:
            r: LoRA Rang
            lora_alpha: LoRA Alpha-Parameter
            lora_dropout: LoRA Dropout-Rate
            target_modules: Liste der Ziel-Module für LoRA
            bias: Bias-Konfiguration für LoRA
        """
        try:
            if self.has_adapters:
                self._remove_adapters()
            
            # Standard Target-Module falls nicht angegeben
            if target_modules is None:
                target_modules = [
                    'gate_proj', 'down_proj', 'v_proj', 'k_proj', 
                    'q_proj', 'o_proj', 'up_proj', 'lm_head'
                ]
            
            # LoRA-Konfiguration erstellen
            self.lora_config = LoraConfig(
                r=r,
                lora_alpha=lora_alpha,
                target_modules=target_modules,
                lora_dropout=lora_dropout,
                bias=bias,
                task_type="SEQ_CLS"
            )
            
            # LoRA-Adapter zum Modell hinzufügen
            self.model = get_peft_model(self.base_model, self.lora_config)
            self.has_adapters = True
            
        except Exception as e:
            raise
    
    def reset_adapter_weights(self) -> None:
        """Setzt die Gewichte der LoRA-Adapter zurück."""
        if not self.has_adapters:
            return
        
        try:
            # Adapter-Gewichte neu initialisieren
            for name, module in self.model.named_modules():
                if hasattr(module, 'reset_parameters') and 'lora' in name.lower():
                    module.reset_parameters()
            
        except Exception as e:
            raise
    
    def reset_model_completely(self) -> None:
        """Setzt das komplette Modell auf den ursprünglichen Zustand zurück."""
        try:
            # Adapter entfernen falls vorhanden
            if self.has_adapters:
                self._remove_adapters()
            
            # GPU-Speicher leeren BEVOR neues Modell geladen wird
            self._clear_gpu_memory()
            
            # Basis-Modell komplett neu laden
            self._load_base_model()
            
        except Exception as e:
            raise
    
    def _clear_gpu_memory(self) -> None:
        """Leert den GPU-Speicher ordnungsgemäß."""
        # Aktuelle Modell-Referenzen löschen
        if hasattr(self, 'model') and self.model is not None:
            del self.model
            
        if hasattr(self, 'base_model') and self.base_model is not None:
            del self.base_model
            
        # Original state dict löschen (wird neu erstellt)
        if hasattr(self, 'original_state_dict') and self.original_state_dict is not None:
            del self.original_state_dict
            
        # Python Garbage Collection
        gc.collect()
        
        # CUDA Cache leeren
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    
    def _remove_adapters(self) -> None:
        """Entfernt LoRA-Adapter vom Modell."""
        if self.has_adapters and hasattr(self.model, 'unload_adapter'):
            try:
                self.model = self.model.unload_adapter()
                self.model = self.base_model
                self.has_adapters = False
                self.lora_config = None
                
                # GPU-Speicher nach Adapter-Entfernung leeren
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                    
            except Exception as e:
                # Fallback: Basis-Modell neu laden
                self.model = self.base_model
                self.has_adapters = False
                self.lora_config = None
    
    def clear_gpu_memory(self) -> None:
        """Öffentliche Methode zum manuellen Leeren des GPU-Speichers."""
        self._clear_gpu_memory()
    
    def get_model(self):
        """Gibt das aktuelle Modell zurück."""
        return self.model
    
    def get_trainable_parameters(self) -> Dict[str, Any]:
        """Gibt Informationen über trainierbare Parameter zurück."""
        if not self.has_adapters:
            total_params = sum(p.numel() for p in self.model.parameters())
            trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        else:
            total_params = sum(p.numel() for p in self.model.parameters())
            trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        
        return {
            "total_parameters": total_params,
            "trainable_parameters": trainable_params,
            "trainable_percentage": 100 * trainable_params / total_params,
            "has_adapters": self.has_adapters,
            "lora_config": self.lora_config.__dict__ if self.lora_config else None
        }
    
    def print_model_info(self) -> None:
        """Druckt Informationen über das aktuelle Modell."""
        info = self.get_trainable_parameters()
        
        print(f"\n{'='*50}")
        print(f"MODEL INFORMATION")
        print(f"{'='*50}")
        print(f"Model Name: {self.model_name}")
        print(f"Has LoRA Adapters: {info['has_adapters']}")
        print(f"Total Parameters: {info['total_parameters']:,}")
        print(f"Trainable Parameters: {info['trainable_parameters']:,}")
        print(f"Trainable %: {info['trainable_percentage']:.2f}%")
        
        if info['lora_config']:
            print(f"\nLoRA Configuration:")
            for key, value in info['lora_config'].items():
                print(f"  {key}: {value}")
        print(f"{'='*50}\n")
    
    def save_model(self, save_path: str) -> None:
        """Speichert das aktuelle Modell."""
        try:
            self.model.save_pretrained(save_path)
        except Exception as e:
            raise
    
    def __repr__(self) -> str:
        return (f"ModelHandler(model_name='{self.model_name}', "
                f"has_adapters={self.has_adapters}, "
                f"num_labels={self.num_labels})")

In [ ]:
# Cross-Validation Script
handler = ModelHandler(model_name=model_name_str)

# 5-Fold Stratified Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Storage lists
metrics_per_fold = []
predictions_per_fold = []

for fold, (train_idx, test_idx) in enumerate(skf.split(df, df['label'])):
    start_time = time.time()
    print(f"Starting Fold {fold + 1}")
    
    # WICHTIG: Frische Adapter für jede Fold
    handler.reset_model_completely()  # Vollständiger Reset
    #handler.add_lora_adapters(r=16, lora_alpha=32, lora_dropout=0.1) ## Longformer without adapters
    handler.print_model_info()
    
    # Prepare train/test splits
    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)
    train_dataset = Dataset.from_pandas(train_df)
    test_dataset = Dataset.from_pandas(test_df)
    tokenized_train = train_dataset.map(preprocess_function, batched=True)
    tokenized_test = test_dataset.map(preprocess_function, batched=True)
    
    # Define output directory for the fold
    output_dir = f"{output_folder}/model_output_fold_{fold + 1}"
    os.makedirs(output_dir, exist_ok=True)
    print('saving in', output_dir)

      # ### Weighting - added from GEMMA_CV_FineTuning_Classification_weighted notebook
    
    ### Class weight
    class_weights = compute_class_weight('balanced', classes=np.unique(tokenized_train['label']), y=tokenized_train['label'])
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
    ### Modify loss function - defined below - unneeded
    #loss_function = CrossEntropyLoss(weight=class_weights_tensor) 
    ### weighted training function
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, num_items_in_batch=None, return_outputs=False):
            labels = inputs.get("labels")
            ### Get the device from the model
            device = model.device
            ### Ensure the class weights are on the same device as the model
            loss_function = CrossEntropyLoss(weight=class_weights_tensor.to(device))
            ### Get model outputs
            outputs = model(**inputs)
            logits = outputs.get("logits")
            ### Calculate the loss with class weights
            loss = loss_function(logits, labels)
            return (loss, outputs) if return_outputs else loss
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        gradient_checkpointing=False,
        learning_rate=learning_rate,
        per_device_train_batch_size=2,
        dataloader_num_workers=2,
        per_device_eval_batch_size=2,
        num_train_epochs=num_train_epochs,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        logging_dir=f"{output_dir}/logs",
        logging_strategy="steps",
        logging_steps=10,
        gradient_accumulation_steps=8,
        bf16=True,
        warmup_ratio=0.05,
        report_to='none',
    )
    
    # Trainer - Neuer Trainer für jede Fold
    trainer = WeightedTrainer( #Careful: need to change from 'Trainer' to use previously defined weighted trainer!
        model=handler.get_model(),  # Verwende get_model() Methode
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        #tokenizer=tokenizer, # no longer compatible with v5 of transformers
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[
        EarlyStoppingCallback(
                early_stopping_patience=early_stop_patience, 
                early_stopping_threshold=early_stop_threshold
            )
        ]
    )
    
    # Train the model
    trainer.train()
    trainer.save_model(output_dir)
    
    # Evaluate on test set
    metrics = trainer.evaluate(tokenized_test)
    print(f"Fold {fold + 1} Metrics: {metrics}")
    
    # Predict on the test set
    predictions = trainer.predict(tokenized_test)
    probabilities_logits = predictions.predictions
    probabilities = torch.nn.functional.softmax(torch.tensor(probabilities_logits), dim=-1).numpy()
    test_df["Probabilities"] = probabilities.tolist()
    test_df["Predicted_Label"] = predictions.predictions.argmax(axis=-1)
    test_df["Fold"] = fold
    
    # Add fold info & time tracking
    fold_metrics = {
        'Fold': fold,
        'Time_Taken_Seconds': time.time() - start_time
    }
    fold_metrics.update(metrics)
    print(fold_metrics)
    
    # Log metrics and predictions
    metrics_per_fold.append(fold_metrics)
    predictions_per_fold.append(test_df.copy())
    
    # WICHTIG: Cleanup nach jeder Fold
    del trainer  # Trainer explizit löschen
    torch.cuda.empty_cache()  # GPU-Cache leeren
    
# Convert to DataFrames
metrics_df = pd.DataFrame(metrics_per_fold)
predictions_df = pd.concat(predictions_per_fold, ignore_index=True)

# Save as Parquet
metrics_df.to_parquet(f"{output_folder}/cross_validation_metrics.pq", engine="pyarrow", index=False)
predictions_df.to_parquet(f"{output_folder}/cross_validation_predictions.pq", engine="pyarrow", index=False)

In [ ]:
# Show summary statistics
filter_cols = ["F1 score", "Precision", "Recall"] #'Time_Taken_Seconds'
df.rename(columns={"eval_f1":"F1 score", 
                           "eval_precision":"Precision", 
                           "eval_recall":"Recall"}, 
                  inplace=True)

# Non filtered
df_mean_std = df[filter_cols].agg(["mean", "std"]).reset_index()
df_mean_std = df_mean_std.round(3) #2
df_mean_std